In [1]:
# Purpose:
# 1. Train classical ML models using identical splits and features as the 1D-CNN
# 2. Perform hyperparameter tuning via cross-validation on training set only
# 3. Evaluate models on internal test set and independent external CRLM cohort
# 4. Ensure fair, reproducible benchmark comparison

print("--- Starting advanced machine learning benchmark WITH hyperparameter tuning ---")

--- Starting advanced machine learning benchmark WITH hyperparameter tuning ---


In [2]:
# --- Import necessary libraries ---
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import (
    train_test_split,
    GridSearchCV,
    StratifiedKFold
)
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    roc_auc_score,
    accuracy_score,
    confusion_matrix,
    precision_score,
    recall_score
)
from sklearn.model_selection import RandomizedSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import (
    RandomForestClassifier,
    GradientBoostingClassifier,
    AdaBoostClassifier
)
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier

import lightgbm as lgb
import xgboost as xgb

warnings.filterwarnings("ignore")

e:\conda-envs\tensorflow\lib\site-packages\scipy\__init__.py:146: UserWarning: A NumPy version >=1.16.5 and <1.23.0 is required for this version of SciPy (detected version 1.24.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


In [3]:
# --- Step 0: Global settings ---
print("\n--- Step 0: Global settings ---")

SEED = 42
np.random.seed(SEED)

plt.rcParams["font.sans-serif"] = ["SimHei", "Arial Unicode MS"]
plt.rcParams["axes.unicode_minus"] = False

BASE_DIR = r"D:\结直肠癌肝转移Biomarker 诊断\新的策略\Autoencoder"
TRAIN_DATA_DIR = BASE_DIR
VALIDATION_DATA_DIR = os.path.join(BASE_DIR, "validation_datasets")
OUTPUT_DIR = os.path.join(BASE_DIR, "advanced_ml_benchmark_results_TUNED")
os.makedirs(OUTPUT_DIR, exist_ok=True)

EXPRESSION_FILE = os.path.join(TRAIN_DATA_DIR, "expression_data_combat_corrected.csv")
METADATA_FILE = os.path.join(TRAIN_DATA_DIR, "metadata_combined.csv")
FUNC_GENES_FILE = os.path.join(TRAIN_DATA_DIR, "functional_genes_620.txt")
EXTERNAL_VALIDATION_FILE = os.path.join(VALIDATION_DATA_DIR, "dat_crlm.csv")

print(f"Results will be saved to: {OUTPUT_DIR}")


--- Step 0: Global settings ---
Results will be saved to: D:\结直肠癌肝转移Biomarker 诊断\新的策略\Autoencoder\advanced_ml_benchmark_results_TUNED


In [4]:
# --- Step 1: Load training & internal test data ---
print("\n--- Step 1: Load and prepare training and internal test data ---")

expression_data = pd.read_csv(EXPRESSION_FILE, index_col=0)
metadata = pd.read_csv(METADATA_FILE, index_col=0)

with open(FUNC_GENES_FILE, "r", encoding="utf-8") as f:
    functional_genes = [line.strip() for line in f if line.strip()]

available_genes = [g for g in functional_genes if g in expression_data.columns]

X = expression_data[available_genes]
y_raw = metadata.reindex(X.index)["group"]

label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y_raw.astype(str))

print("Label mapping:", dict(zip(label_encoder.classes_,
                                 label_encoder.transform(label_encoder.classes_))))

X_train, X_test_internal, y_train, y_test_internal = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=SEED
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_internal_scaled = scaler.transform(X_test_internal)

print(f"Training samples: {X_train.shape[0]}")
print(f"Internal test samples: {X_test_internal.shape[0]}")


--- Step 1: Load and prepare training and internal test data ---
Label mapping: {'metastasis': 0, 'primary': 1}
Training samples: 861
Internal test samples: 216


In [5]:
# --- Step 2: Load external CRLM validation data ---
print("\n--- Step 2: Load and prepare external validation data ---")

dat_crlm = pd.read_csv(EXTERNAL_VALIDATION_FILE, index_col=0)

X_val_external = pd.DataFrame(index=dat_crlm.index, columns=available_genes)
for g in available_genes:
    X_val_external[g] = dat_crlm[g] if g in dat_crlm.columns else 0.0

y_val_external = dat_crlm["status"].apply(
    lambda x: 1 if "metastasis" in str(x).lower() else 0
).astype(int)

X_val_external_scaled = scaler.transform(X_val_external)

print(f"External validation samples: {X_val_external.shape[0]}")


--- Step 2: Load and prepare external validation data ---
External validation samples: 36


In [6]:
# --- Step 3: Hyperparameter tuning ---
print("\n--- Step 3: Hyperparameter tuning via 5-fold CV ---")

cv_strategy = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=SEED
)

# ---- Logistic Regression ----
lr_grid = GridSearchCV(
    LogisticRegression(max_iter=2000, random_state=SEED),
    param_grid={"C": [0.01, 0.1, 1, 10]},
    scoring="roc_auc",
    cv=cv_strategy,
    n_jobs=-1
)
lr_grid.fit(X_train_scaled, y_train)
best_lr = lr_grid.best_estimator_
print("Best LR params:", lr_grid.best_params_)

# ---- Random Forest ----
rf_grid = GridSearchCV(
    RandomForestClassifier(random_state=SEED, n_jobs=-1),
    param_grid={
        "n_estimators": [200, 500],
        "max_depth": [None, 10, 20],
        "min_samples_leaf": [1, 2]
    },
    scoring="roc_auc",
    cv=cv_strategy,
    n_jobs=-1
)
rf_grid.fit(X_train_scaled, y_train)
best_rf = rf_grid.best_estimator_
print("Best RF params:", rf_grid.best_params_)

# ---- XGBoost ----
xgb_grid = GridSearchCV(
    xgb.XGBClassifier(
        random_state=SEED,
        eval_metric="logloss",
        use_label_encoder=False
    ),
    param_grid={
        "n_estimators": [200, 500],
        "max_depth": [3, 6],
        "learning_rate": [0.01, 0.1],
        "subsample": [0.8, 1.0]
    },
    scoring="roc_auc",
    cv=cv_strategy,
    n_jobs=-1
)
xgb_grid.fit(X_train_scaled, y_train)
best_xgb = xgb_grid.best_estimator_
print("Best XGB params:", xgb_grid.best_params_)

# ---- LightGBM ----
best_lgb = lgb.LGBMClassifier(
    random_state=SEED,
    n_estimators=200,
    learning_rate=0.05,
    num_leaves=31
)


--- Step 3: Hyperparameter tuning via 5-fold CV ---
Best LR params: {'C': 0.1}
Best RF params: {'max_depth': 20, 'min_samples_leaf': 1, 'n_estimators': 500}
Best XGB params: {'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 500, 'subsample': 0.8}


In [7]:
# --- Step 4: Define final model list ---
models = {
    "Logistic Regression": best_lr,
    "Random Forest": best_rf,
    "XGBoost": best_xgb,
    "LightGBM": best_lgb,
    "Gradient Boosting": GradientBoostingClassifier(random_state=SEED),
    "AdaBoost": AdaBoostClassifier(random_state=SEED),
    "Support Vector Machine": SVC(probability=True, random_state=SEED),
    "K-Nearest Neighbors": KNeighborsClassifier(),
    "Decision Tree": DecisionTreeClassifier(random_state=SEED),
    "Gaussian Naive Bayes": GaussianNB()
}


In [8]:
# --- Step 5: Evaluation function ---
def evaluate_model(y_true, y_proba, y_pred):
    auc = roc_auc_score(y_true, y_proba) if len(np.unique(y_true)) > 1 else np.nan
    acc = accuracy_score(y_true, y_pred)
    cm = confusion_matrix(y_true, y_pred)
    if cm.shape == (2, 2):
        tn, fp, fn, tp = cm.ravel()
        sens = tp / (tp + fn) if (tp + fn) > 0 else np.nan
        spec = tn / (tn + fp) if (tn + fp) > 0 else np.nan
    else:
        sens, spec = np.nan, np.nan
    return {
        "AUC": auc,
        "Accuracy": acc,
        "Sensitivity": sens,
        "Specificity": spec,
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0)
    }

In [9]:
# --- Step 6: Run evaluation ---
print("\n--- Step 6: Evaluating models ---")

internal_results = []
external_results = []

for name, model in models.items():
    print(f"Evaluating {name}")
    model.fit(X_train_scaled, y_train)

    y_proba_int = model.predict_proba(X_test_internal_scaled)[:, 1]
    y_pred_int = model.predict(X_test_internal_scaled)
    res_int = evaluate_model(y_test_internal, y_proba_int, y_pred_int)
    res_int["Model"] = name
    internal_results.append(res_int)

    y_proba_ext = model.predict_proba(X_val_external_scaled)[:, 1]
    y_pred_ext = model.predict(X_val_external_scaled)
    res_ext = evaluate_model(y_val_external, y_proba_ext, y_pred_ext)
    res_ext["Model"] = name
    external_results.append(res_ext)

internal_df = pd.DataFrame(internal_results).sort_values("AUC", ascending=False)
external_df = pd.DataFrame(external_results).sort_values("AUC", ascending=False)

internal_df.to_csv(os.path.join(OUTPUT_DIR, "internal_benchmark_TUNED.csv"), index=False)
external_df.to_csv(os.path.join(OUTPUT_DIR, "external_benchmark_TUNED.csv"), index=False)

print("\n✅ Benchmark completed with hyperparameter tuning")


--- Step 6: Evaluating models ---
Evaluating Logistic Regression
Evaluating Random Forest
Evaluating XGBoost
Evaluating LightGBM
[LightGBM] [Info] Number of positive: 574, number of negative: 287
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.008788 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 154479
[LightGBM] [Info] Number of data points in the train set: 861, number of used features: 606
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.666667 -> initscore=0.693147
[LightGBM] [Info] Start training from score 0.693147
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
Evaluating Gradient Boosting
Evaluating AdaBoost
Evaluating Support Vector Machine
Evaluating K-Nearest Neighbors
Evaluating Decision Tree
Evaluating Gaussian Naive Bayes

✅ Benchmark completed with hyperparameter tuning
